In [ ]:
import requests
from bs4 import BeautifulSoup
import re, json, time, random

N_ARTICLES = 500  # bump this up once both versions check out

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
}

def extract_categories(html):
    """Pull all franchise-type tags (real categories) — excludes person/keyword tags."""
    tag_objs = re.findall(r'\{"headline":"[^{}]*"__typename":"tag"\}', html)
    cats = {}
    for raw in tag_objs:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if obj.get("type") == "franchise":
            cats[obj["id"]] = obj.get("tagName") or obj.get("headline")
    return list(cats.values())

def parse_article(url, html):
    soup = BeautifulSoup(html, "html.parser")

    ld_script = soup.find("script", type="application/ld+json")
    ld = {}
    if ld_script and ld_script.string:
        try:
            ld = json.loads(ld_script.string)
        except json.JSONDecodeError:
            pass

    title = ld.get("headline", "")
    authors = "; ".join(a.get("name", "") for a in ld.get("author", []) if isinstance(a, dict))
    date_published = ld.get("datePublished", "")
    date_modified = ld.get("dateModified", "")

    body_div = soup.find("div", class_="ArticleBody-articleBody")
    content = ""
    if body_div:
        paragraphs = body_div.find_all("p")
        content = "\n".join(p.get_text(" ", strip=True) for p in paragraphs)

    categories = extract_categories(html)

    return {
        "url": url,
        "title": title,
        "author": authors,
        "date_published": date_published,
        "date_modified": date_modified,
        "categories": categories,   # kept as a list here — each version below formats it differently
        "content": content,
    }

def fetch_and_parse(url):
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        if resp.status_code != 200:
            return None
        return parse_article(url, resp.text)
    except Exception as e:
        print(f"failed: {url} ({e})")
        return None

In [ ]:
import csv
import pandas as pd

sitemap_df = pd.read_csv("merged_sitemaps.csv")  # adjust filename — needs a 'loc' column
urls_to_scrape = sitemap_df["loc"].head(N_ARTICLES).tolist()

CSV_OUTPUT = "cnbc_articles_500.csv"

with open(CSV_OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(["url", "title", "author", "date_published", "date_modified", "categories", "content"])

    for i, url in enumerate(urls_to_scrape, 1):
        result = fetch_and_parse(url)
        if result:
            writer.writerow([
                result["url"],
                result["title"],
                result["author"],
                result["date_published"],
                result["date_modified"],
                "; ".join(result["categories"]),  # multi-category joined into one string
                result["content"],
            ])
            f.flush()
        print(f"{i}/{len(urls_to_scrape)} done: {url}")
        time.sleep(random.uniform(0.5, 1.5))  # polite random delay

print(f"\nSaved to {CSV_OUTPUT}")
pd.read_csv(CSV_OUTPUT)

1/20 done: https://www.cnbc.com/2021/03/08/pentagon-uncertain-of-troop-withdrawal-in-afghanistan-as-deadline-looms-.html
2/20 done: https://www.cnbc.com/2021/03/08/stocks-making-the-biggest-moves-after-the-bell-zoom-video-stitch-fix-invitae.html
3/20 done: https://www.cnbc.com/2021/03/08/cathie-wood-says-she-is-still-bullish-on-tesla-hints-at-a-new-price-target.html
4/20 done: https://www.cnbc.com/2021/03/08/cathie-wood-sees-bitcoin-joining-stocks-and-bonds-as-part-of-the-classic-investor-allocation-model.html
5/20 done: https://www.cnbc.com/2021/03/08/covid-melinda-gates-says-we-could-reach-global-herd-immunity-sometime-in-2022.html
6/20 done: https://www.cnbc.com/2021/03/08/forex-markets-bonds-moves-risk-currencies-and-dollar.html
7/20 done: https://www.cnbc.com/2021/03/08/us-bonds-treasury-yields-rise-after-senate-passes-stimulus-package.html
8/20 done: https://www.cnbc.com/2021/03/07/stock-market-open-to-close-news.html
9/20 done: https://www.cnbc.com/2021/03/08/stocks-making-the-b

,url,title,author,date_published,date_modified,categories,content
0,https://www.cnbc.com/2021/03/08/pentagon-uncer...,Pentagon uncertain of U.S. troop withdrawal in...,Amanda Macias,2021-03-08T20:59:00+0000,2021-03-08T22:29:46+0000,Politics; US: News; Defense; US Top News and A...,WASHINGTON — The Pentagon said Monday that it ...
1,https://www.cnbc.com/2021/03/08/stocks-making-...,Stocks making the biggest moves after the bell...,Rich Mendez,2021-03-08T22:16:12+0000,2021-03-08T22:16:12+0000,Finance; stocks; Market Insider; Economy; Markets,Check out the companies making headlines after...
2,https://www.cnbc.com/2021/03/08/cathie-wood-sa...,Cathie Wood says she is still bullish on Tesla...,Kevin Stankiewicz,2021-03-08T21:05:37+0000,2021-03-08T22:08:02+0000,stocks; Investing; Autos; Closing Bell Parent,NaN
3,https://www.cnbc.com/2021/03/08/cathie-wood-se...,Cathie Wood sees bitcoin joining stocks and bo...,Jesse Pound,2021-03-08T21:35:34+0000,2021-03-08T21:43:18+0000,Investing; Cryptocurrency; Bitcoin; Wall Stree...,Bitcoin and other cryptocurrencies could event...
4,https://www.cnbc.com/2021/03/08/covid-melinda-...,Melinda Gates says we could reach global herd ...,Berkeley Lovelace Jr.,2021-03-08T21:36:09+0000,2021-03-08T21:36:09+0000,Coronavirus; Politics; World News; Biotech and...,Billionaire philanthropist and former tech exe...
5,https://www.cnbc.com/2021/03/08/forex-markets-...,Dollar at three-and-a-half-month high on firme...,NaN,2021-03-08T06:26:43+0000,2021-03-08T21:34:58+0000,EU FX; Asia FX; Americas FX; Currencies,The U.S. dollar hit a 3-1/2-month high against...
6,https://www.cnbc.com/2021/03/08/us-bonds-treas...,10-year Treasury yield rises to roughly 1.6% a...,Vicky McKeever,2021-03-08T08:52:06+0000,2021-03-08T21:28:29+0000,Markets; US Economy; World Markets; Bonds; Cen...,The 10-year U.S. Treasury yield traded near th...
7,https://www.cnbc.com/2021/03/07/stock-market-o...,"Dow rises 300 points to touch a record, Nasdaq...",Yun Li; Jesse Pound,2021-03-07T23:05:29+0000,2021-03-08T21:25:17+0000,U.S. Markets; Investing; Index ETFs; Bonds; Ec...,The Dow Jones Industrial Average climbed on Mo...
8,https://www.cnbc.com/2021/03/08/stocks-making-...,Stocks making the biggest moves midday: Apollo...,Jesse Pound,2021-03-08T17:12:15+0000,2021-03-08T21:24:52+0000,Investing; U.S. Markets; Wall Street; Finance;...,Here are the stocks making headlines in midday...
9,https://www.cnbc.com/2021/03/08/cathie-wood-na...,Cathie Wood names one of the most underappreci...,Pippa Stevens,2021-03-08T21:13:06+0000,2021-03-08T21:18:06+0000,stocks; Investing; Closing Bell,NaN
